# 03｜TensorFlow：CNN、GRU 與預訓練模型入口

對照原筆記的卷積／池化／GAP、序列模型與遷移學習。預設不下載資料或預訓練權重；`weights=None` 只檢查架構，**不算遷移學習**。

In [ ]:
import numpy as np
import tensorflow as tf

tf.keras.utils.set_random_seed(42)
rng = np.random.default_rng(42)
images = rng.random((64, 16, 16, 1), dtype=np.float32)
labels = (images[:, :, :8, :].mean(axis=(1, 2, 3)) >
          images[:, :, 8:, :].mean(axis=(1, 2, 3))).astype('int32')
cnn = tf.keras.Sequential([
    tf.keras.Input(shape=(16, 16, 1)),
    tf.keras.layers.Conv2D(8, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(2),
])
cnn.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
print('CNN:', cnn(images[:4]).shape)
cnn.fit(images, labels, epochs=1, batch_size=16, verbose=0)
assert cnn(images[:4]).shape == (4, 2)

TensorFlow 影像預設 `[N,H,W,C]`；PyTorch 對應 `[N,C,H,W]`。隨機增強只放訓練路徑，且需先確認翻轉、旋轉不改變標籤意義。

In [ ]:
seq = rng.normal(size=(64, 6, 2)).astype('float32')
seq_y = (seq[:, -1, 0] > 0).astype('float32')[:, None]
gru_model = tf.keras.Sequential([
    tf.keras.Input(shape=(None, 2)), tf.keras.layers.GRU(12), tf.keras.layers.Dense(1),
])
gru_model.compile(optimizer='adam', loss=tf.keras.losses.BinaryCrossentropy(from_logits=True))
print('GRU:', gru_model(seq[:4]).shape)
gru_model.fit(seq, seq_y, epochs=1, batch_size=16, verbose=0)
assert gru_model(seq[:4]).shape == (4, 1)

真實時間序列不能像這份獨立的合成資料那樣任意切分；要依時間保留驗證與測試期。若序列長度不同，需 padding/mask。疊兩層 RNN/GRU 時，前層設 `return_sequences=True`。

In [ ]:
# 離線只檢查網路連接；沒有預訓練權重。
inputs = tf.keras.Input(shape=(96, 96, 3))
base = tf.keras.applications.MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights=None)
base.trainable = False
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)  # 期待 0–255 像素
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
outputs = tf.keras.layers.Dense(2)(x)
transfer_structure = tf.keras.Model(inputs, outputs)
print('backbone output:', transfer_structure(tf.zeros((1, 96, 96, 3))).shape)
assert transfer_structure(tf.zeros((1, 96, 96, 3))).shape == (1, 2)

真正遷移學習時改用 `weights='imagenet'`，首次載入會下載權重；先凍結 backbone 訓練新分類頭，再以低學習率微調並重新 `compile`。若 backbone 含 BatchNorm，微調時仍以 `base(x, training=False)` 保持其推論統計。詳見 [TensorFlow 工具書](../../guide/tensorflow.md) 與 [雙框架對照](../../guide/framework-compare.md)。